# ⚽ Pipeline de Análise de Futebol no Google Colab

Este notebook usa o projeto `poc-analyze-ia` (zipado) para:

- Configurar ambiente com Ultralytics YOLOv8m
- Validar o dataset de homografia `football-field-detection.v15i.yolov8`
- Treinar rapidamente um modelo de detecção em `Soccer-1`
- (Opcional) Processar um vídeo com o detector

**Pré-requisito**: fazer upload do arquivo `poc-analyze-ia.zip` (gerado da sua máquina) para o Colab na célula abaixo.

In [ ]:
# ✅ 1. Verificar arquivos enviados ao Colab
# Faça upload manual do arquivo `poc-analyze-ia.zip` usando o painel de arquivos à esquerda
# (ícone de pasta > botão Upload).

!ls -lh

In [ ]:
# ✅ 2. Descompactar o projeto e entrar na pasta

!unzip -q poc-analyze-ia.zip
%cd poc-analyze-ia

!ls

In [ ]:
# ✅ 3. Instalar dependências do projeto

!pip install --upgrade pip
!pip install -r requirements.txt

# Garantir ultralytics, supervision e roboflow (caso não venham juntos)
!pip install ultralytics supervision roboflow

In [ ]:
# ✅ 4. Verificar GPU do Colab

import torch, platform
print("Python:", platform.python_version())
print("CUDA disponível:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU atual:", torch.cuda.get_device_name(0))
else:
    print("Sem GPU ativa. Vá em Runtime > Change runtime type > GPU.")

## 🔍 5. Validação rápida do dataset de Homografia (pose)

Roda `src/test_football_field_detection.py`, que faz apenas **validação** do dataset de homografia `football-field-detection.v15i.yolov8` usando `yolov8m-pose.pt` com batch/resolução reduzidos.

In [ ]:
%cd /content/poc-analyze-ia

!python src/test_football_field_detection.py

## 🧠 6. Treino rápido de detecção em `Soccer-1`

Treina um modelo YOLOv8m de detecção usando o dataset `Soccer-1/data.yaml` por poucas épocas, só para validar o fluxo.

- Se quiser usar outro dataset, troque o caminho do `data_yaml`.
- O código já escolhe GPU (`device=0`) se estiver disponível.

In [ ]:
from src.train_custom_model import train_football_model
from pathlib import Path
import torch

base_dir = Path(".")
data_yaml = base_dir / "Soccer-1" / "data.yaml"  # ajuste se quiser outro dataset

device = 0 if torch.cuda.is_available() else "cpu"
print("Usando device:", device)
print("data.yaml:", data_yaml)

results = train_football_model(
    dataset_path=str(data_yaml),
    epochs=5,        # teste leve
    imgsz=640,
    batch_size=4,
    device=device,
    name="soccer1_colab",
)

results

## 🎬 7. (Opcional) Processar um vídeo com o detector

Fluxo:
1. Faça upload de um vídeo de futebol pelo painel de arquivos (por exemplo `video_futebol.mp4` em `/content`).
2. Use a célula abaixo para rodar o detector e salvar um vídeo anotado em `videos/resultado_colab.mp4`.
3. Faça download do resultado para visualizar localmente.

In [ ]:
from pathlib import Path
from football_detector import FootballDetector

base_dir = Path(".")
VIDEOS_DIR = base_dir / "videos"
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

# Caminho do vídeo enviado (ajuste o nome se necessário)
video_path = "/content/video_futebol.mp4"  # faça upload desse arquivo antes
output_path = str(VIDEOS_DIR / "resultado_colab.mp4")

detector = FootballDetector(
    model_path="yolov8m.pt",
    conf_threshold=0.4,
)

detector.process_video(
    video_path=video_path,
    output_path=output_path,
    display=False,  # importante: não tentar abrir janela no Colab
)

print("Vídeo processado salvo em:", output_path)

## ⬇️ 8. Download do vídeo anotado

Baixar o `resultado_colab.mp4` para sua máquina local.

In [ ]:
from google.colab import files
from pathlib import Path

output_path = Path("videos") / "resultado_colab.mp4"

if output_path.exists():
    files.download(str(output_path))
else:
    print("Arquivo de saída não encontrado:", output_path)